# TensorForge 2.0: multilingual encoder for ticket routing

One shared multilingual encoder with three heads: category (11), secondary (incl. "none"), urgent.

**Run 1** trains on `train.csv` and evaluates on `validation.csv` every epoch. This is the number we report.
**Run 2** retrains on train + validation for the best epoch count and is exported to ONNX int8 for the API.

Kaggle setup: GPU T4 x2 or P100, Internet ON, and the organisers' CSVs added as a dataset.

In [ ]:
import json, math, os, random, re, time, unicodedata, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

TINY = os.environ.get("TINY") == "1"  # smoke test with a tiny random model (no download)

CFG = dict(
    model_name="intfloat/multilingual-e5-base",   # try also: xlm-roberta-base, sentence-transformers/LaBSE
    max_len=384,
    batch_size=16,
    epochs=6,
    lr_encoder=3e-5,
    lr_heads=1e-3,
    weight_decay=0.01,
    warmup=0.1,
    label_smoothing=0.05,
    w_sec=0.5,
    w_urg=0.5,
    aug_prob=0.5,
    seed=42,
    final_fit=True,
)
if TINY:
    CFG.update(epochs=2, max_len=64, batch_size=8)


def find_data_dir():
    for p in [os.environ.get("DATA_DIR"), "/kaggle/input", "data", "../data"]:
        if p and Path(p).exists():
            hits = list(Path(p).rglob("train.csv"))
            if hits:
                return hits[0].parent
    raise FileNotFoundError("train.csv not found. Add the dataset to the notebook.")


DATA_DIR = find_data_dir()
OUT = Path(os.environ.get("OUT_DIR", "/kaggle/working/encoder_export" if Path("/kaggle/working").exists() else "encoder_export"))
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("data:", DATA_DIR, "| out:", OUT, "| device:", DEVICE)


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


seed_all(CFG["seed"])

## Data
The model sees only channel, subject and text. This must match `app/textprep.py` in the repo exactly.

In [ ]:
_WS = re.compile(r"\s+")


def normalize(s):
    s = unicodedata.normalize("NFC", s or "")
    return _WS.sub(" ", s).strip().lower()


def model_input(channel, subject, text):
    return f"[{channel}] {normalize(subject)} || {normalize(text)}"


def load(name):
    d = pd.read_csv(DATA_DIR / f"{name}.csv", keep_default_na=False, dtype=str)
    d["sec"] = d.secondary_category.replace("", "none")
    d["urgent"] = (d.is_urgent.str.lower() == "true").astype(int)
    return d


tr, va = load("train"), load("validation")
CATS = sorted(tr.category.unique())
SECS = ["none"] + sorted(s for s in tr.sec.unique() if s != "none")
cat2i = {c: i for i, c in enumerate(CATS)}
sec2i = {s: i for i, s in enumerate(SECS)}
print(len(tr), len(va), CATS, SECS, sep="\n")

## Augmentation (train only)
Light, label-preserving noise so the model does not lean on sentence order or exact filler lines.

In [ ]:
_SPLIT = re.compile(r"(?<=[.!?\n])\s+")


def augment(channel, subject, text):
    if random.random() > CFG["aug_prob"]:
        return channel, subject, text
    parts = [p for p in _SPLIT.split(text) if p.strip()]
    if len(parts) > 2:
        if random.random() < 0.5:  # drop one sentence, never all
            parts.pop(random.randrange(len(parts)))
        if random.random() < 0.3:
            random.shuffle(parts)
        text = " ".join(parts)
    if subject and random.random() < 0.2:
        subject = ""
    if random.random() < 0.1:
        text = text.upper()
    return channel, subject, text


class Tickets(Dataset):
    def __init__(self, df, train):
        self.df, self.train = df.reset_index(drop=True), train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        ch, sj, tx = r.channel, r.subject, r.text
        if self.train:
            ch, sj, tx = augment(ch, sj, tx)
        return model_input(ch, sj, tx), cat2i[r.category], sec2i[r.sec], r.urgent

## Model

In [ ]:
def load_backbone():
    if TINY:  # random weights + tokenizer trained on our data, so the pipeline runs offline
        from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, processors, trainers
        from transformers import PreTrainedTokenizerFast, XLMRobertaConfig, XLMRobertaModel
        tok = Tokenizer(models.Unigram())
        tok.normalizer = normalizers.NFKC()
        tok.pre_tokenizer = pre_tokenizers.Metaspace()
        tok.train_from_iterator([model_input(*x) for x in zip(tr.channel, tr.subject, tr.text)],
                                trainers.UnigramTrainer(vocab_size=3000, special_tokens=["<s>", "<pad>", "</s>", "<unk>"], unk_token="<unk>"))
        tok.post_processor = processors.TemplateProcessing(single="<s> $A </s>", special_tokens=[("<s>", 0), ("</s>", 2)])
        hf_tok = PreTrainedTokenizerFast(tokenizer_object=tok, bos_token="<s>", eos_token="</s>", unk_token="<unk>", pad_token="<pad>")
        cfg = XLMRobertaConfig(vocab_size=3000, hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
                               intermediate_size=128, max_position_embeddings=CFG["max_len"] + 4, pad_token_id=1)
        return hf_tok, XLMRobertaModel(cfg)
    return AutoTokenizer.from_pretrained(CFG["model_name"]), AutoModel.from_pretrained(CFG["model_name"])


class TicketModel(nn.Module):
    def __init__(self, encoder, n_cat, n_sec):
        super().__init__()
        self.encoder = encoder
        h = encoder.config.hidden_size
        self.drop = nn.Dropout(0.1)
        self.cat = nn.Linear(h, n_cat)
        self.sec = nn.Linear(h, n_sec)
        self.urg = nn.Linear(h, 1)

    def forward(self, input_ids, attention_mask):
        hs = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(hs.dtype)
        pooled = self.drop((hs * m).sum(1) / m.sum(1).clamp(min=1e-6))  # mean pooling
        return self.cat(pooled), self.sec(pooled), self.urg(pooled).squeeze(-1)


tokenizer, _probe = load_backbone()
del _probe


def collate(batch):
    texts, c, s, u = zip(*batch)
    enc = tokenizer(list(texts), max_length=CFG["max_len"], truncation=True, padding=True, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"], torch.tensor(c), torch.tensor(s), torch.tensor(u, dtype=torch.float)


lens = [len(tokenizer(model_input(*x))["input_ids"]) for x in zip(va.channel, va.subject, va.text)]
print("validation token length p50/p95/max:", np.percentile(lens, 50), np.percentile(lens, 95), max(lens),
      f"| truncated: {np.mean(np.array(lens) > CFG['max_len']):.1%}")

## Training and evaluation

In [ ]:
@torch.no_grad()
def predict_probs(model, df, bs=64):
    model.eval()
    dl = DataLoader(Tickets(df, train=False), batch_size=bs, shuffle=False, collate_fn=collate)
    pc, ps, pu = [], [], []
    for ids, mask, *_ in dl:
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            c, s, u = model(ids.to(DEVICE), mask.to(DEVICE))
        pc.append(F.softmax(c.float(), -1).cpu()); ps.append(F.softmax(s.float(), -1).cpu()); pu.append(torch.sigmoid(u.float()).cpu())
    return torch.cat(pc).numpy(), torch.cat(ps).numpy(), torch.cat(pu).numpy()


def evaluate(df, pc, ps, pu, verbose=False):
    cat = np.array(CATS)[pc.argmax(1)]
    sec = np.array(SECS)[ps.argmax(1)]
    urg = (pu >= 0.5).astype(int)
    out = {
        "category_macro_f1": round(f1_score(df.category, cat, average="macro"), 4),
        "category_accuracy": round(float((cat == df.category.values).mean()), 4),
        "secondary_macro_f1": round(f1_score(df.sec, sec, average="macro"), 4),
        "urgent_f1": round(f1_score(df.urgent, urg), 4),
        "accuracy_by_language": {k: round(float(v), 4) for k, v in
                                 pd.Series(cat == df.category.values).groupby(df.language.values).mean().items()},
    }
    if verbose:
        print(classification_report(df.category, cat, digits=3))
    return out


def train_run(train_df, epochs, eval_df=None, tag="run"):
    seed_all(CFG["seed"])
    _, encoder = load_backbone()
    model = TicketModel(encoder, len(CATS), len(SECS)).to(DEVICE)
    dl = DataLoader(Tickets(train_df, train=True), batch_size=CFG["batch_size"], shuffle=True, collate_fn=collate,
                    num_workers=2, drop_last=True)
    head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
    opt = torch.optim.AdamW([{"params": model.encoder.parameters(), "lr": CFG["lr_encoder"]},
                             {"params": head_params, "lr": CFG["lr_heads"]}], weight_decay=CFG["weight_decay"])
    steps = epochs * len(dl)
    sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * steps), steps)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

    sec_counts = train_df.sec.map(sec2i).value_counts().reindex(range(len(SECS)), fill_value=1).values
    sec_w = torch.tensor((sec_counts.sum() / sec_counts) ** 0.5, dtype=torch.float, device=DEVICE)
    sec_w = sec_w / sec_w.mean()
    pos_w = torch.tensor((1 - train_df.urgent.mean()) / train_df.urgent.mean(), device=DEVICE).sqrt()

    history, best = [], (-1, None, 0)
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot = 0.0
        for ids, mask, yc, ys, yu in dl:
            ids, mask, yc, ys, yu = (x.to(DEVICE) for x in (ids, mask, yc, ys, yu))
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                c, s, u = model(ids, mask)
                loss = (F.cross_entropy(c.float(), yc, label_smoothing=CFG["label_smoothing"])
                        + CFG["w_sec"] * F.cross_entropy(s.float(), ys, weight=sec_w)
                        + CFG["w_urg"] * F.binary_cross_entropy_with_logits(u.float(), yu, pos_weight=pos_w))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        row = {"epoch": ep, "train_loss": round(tot / len(dl), 4), "secs": round(time.time() - t0)}
        if eval_df is not None:
            m = evaluate(eval_df, *predict_probs(model, eval_df))
            row.update({k: v for k, v in m.items() if k != "accuracy_by_language"})
            if m["category_macro_f1"] > best[0]:
                best = (m["category_macro_f1"], {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, ep)
        history.append(row); print(tag, row)
    if eval_df is not None:
        model.load_state_dict(best[1])
    return model, history, best[2]

### Run 1: train on train.csv, pick the best epoch on validation

In [ ]:
model, hist1, best_epoch = train_run(tr, CFG["epochs"], eval_df=va, tag="run1")
pc, ps, pu = predict_probs(model, va)
val_metrics = evaluate(va, pc, ps, pu, verbose=True)
print("best epoch:", best_epoch); print(json.dumps(val_metrics, indent=2))
np.savez(OUT / "val_probs.npz", cat=pc, sec=ps, urg=pu)
cm = confusion_matrix(va.category, np.array(CATS)[pc.argmax(1)], labels=CATS)
pd.DataFrame(cm, index=CATS, columns=CATS).to_csv(OUT / "val_confusion.csv")

errs = va.assign(pred=np.array(CATS)[pc.argmax(1)], conf=pc.max(1))
errs = errs[errs.pred != errs.category].sort_values("conf", ascending=False)
errs[["ticket_id", "language", "channel", "category", "pred", "conf", "text"]].to_csv(OUT / "val_errors.csv", index=False)
print(errs[["language", "category", "pred", "conf"]].head(15))

### Run 2: retrain on train + validation for the chosen epoch count (this model ships)

In [ ]:
if CFG["final_fit"]:
    del model; torch.cuda.empty_cache()
    full = pd.concat([tr, va], ignore_index=True)
    model, hist2, _ = train_run(full, best_epoch, tag="run2")
else:
    hist2 = []

## Export to ONNX int8 and check it matches PyTorch

In [ ]:
try:
    import onnx, onnxruntime  # noqa: F401
except ImportError:  # Kaggle images do not ship these
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic


class Export(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m

    def forward(self, input_ids, attention_mask):
        c, s, u = self.m(input_ids, attention_mask)
        return F.softmax(c, -1), F.softmax(s, -1), torch.sigmoid(u)


model.eval().float().cpu()
exp = Export(model).eval()
sample = tokenizer(["[chat] || hello", "[email] subject || a longer second example text"], padding=True, return_tensors="pt")
fp32, int8 = OUT / "model.fp32.onnx", OUT / "model.onnx"
torch.onnx.export(exp, (sample["input_ids"], sample["attention_mask"]), str(fp32), dynamo=False,
                  input_names=["input_ids", "attention_mask"], output_names=["p_cat", "p_sec", "p_urg"],
                  dynamic_axes={"input_ids": {0: "b", 1: "t"}, "attention_mask": {0: "b", 1: "t"},
                                "p_cat": {0: "b"}, "p_sec": {0: "b"}, "p_urg": {0: "b"}}, opset_version=17)
quantize_dynamic(str(fp32), str(int8), weight_type=QuantType.QInt8)
fp32.unlink()
for f in OUT.glob("*.onnx.data"):
    f.unlink()

tokenizer.save_pretrained(OUT / "hf_tokenizer")
(OUT / "tokenizer.json").write_bytes((OUT / "hf_tokenizer" / "tokenizer.json").read_bytes())

# parity: serving uses the `tokenizers` library + onnxruntime, so test exactly that path
from tokenizers import Tokenizer
st = Tokenizer.from_file(str(OUT / "tokenizer.json"))
st.enable_truncation(CFG["max_len"])
pad_id = tokenizer.pad_token_id
st.enable_padding(pad_id=pad_id, pad_token=tokenizer.pad_token)
sess = ort.InferenceSession(str(int8), providers=["CPUExecutionProvider"])
texts = [model_input(*x) for x in zip(va.channel, va.subject, va.text)]
same_ids = all(st.encode(t).ids == tokenizer(t, max_length=CFG["max_len"], truncation=True)["input_ids"] for t in texts[:200])
onnx_cat = []
for i in range(0, len(texts), 32):
    enc = st.encode_batch(texts[i:i + 32])
    ids = np.array([e.ids for e in enc], dtype=np.int64); mask = np.array([e.attention_mask for e in enc], dtype=np.int64)
    onnx_cat.append(sess.run(["p_cat"], {"input_ids": ids, "attention_mask": mask})[0])
onnx_cat = np.concatenate(onnx_cat)
torch_cat, _, _ = predict_probs(model.to(DEVICE), va)
agree = float((onnx_cat.argmax(1) == torch_cat.argmax(1)).mean())
print(f"tokenizer parity: {same_ids} | int8 vs torch argmax agreement: {agree:.4f} | size: {int8.stat().st_size / 1e6:.0f} MB")

## Save everything and zip for download

In [ ]:
meta = {"kind": "onnx_encoder", "model_name": CFG["model_name"], "max_len": CFG["max_len"],
        "pad_id": pad_id, "pad_token": tokenizer.pad_token, "cat_classes": CATS, "sec_classes": SECS,
        "config": CFG, "best_epoch": best_epoch, "val_metrics_run1": val_metrics,
        "history_run1": hist1, "history_run2": hist2, "onnx_int8_agreement": agree, "tokenizer_parity": same_ids}
(OUT / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
short = CFG["model_name"].split("/")[-1]
zpath = OUT.parent / f"encoder_{short}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(OUT.parent))
print("saved", zpath, f"{zpath.stat().st_size / 1e6:.0f} MB")